In [1]:
print("hello")

hello


In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.schema import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional


# ----------- Define State -----------
class ResumeMatchState(TypedDict):
    resume_text: Optional[str]
    job_description: Optional[str]
    structured_resume: Optional[str]
    structured_job: Optional[str]
    match_score: Optional[int]
    match_feedback: Optional[str]
    cover_letter: Optional[str]
    suggestions: Optional[str]

#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

# ----------- LLM Setup -----------



llm = ChatNVIDIA(model="mistralai/mixtral-8x7b-instruct-v0.1",api_key="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")


# llm = ChatOpenAI(model="gpt-4", temperature=0)

# ----------- Nodes -----------

# Node 1: Normalize Resume
def load_resume(state: ResumeMatchState):
    return {"resume_text": state["resume_text"]}  # Already extracted text

# Node 2: Normalize JD
def load_job_description(state: ResumeMatchState):
    return {"job_description": state["job_description"]}

# Node 3: Extract Info
def extract_info(state: ResumeMatchState):
    resume_prompt = f"Extract key skills, tools, and experience from this resume:\n{state['resume_text']}"
    jd_prompt = f"Extract key requirements, skills, and responsibilities from this job description:\n{state['job_description']}"

    structured_resume = llm.invoke([HumanMessage(content=resume_prompt)]).content
    structured_job = llm.invoke([HumanMessage(content=jd_prompt)]).content

    return {
        "structured_resume": structured_resume,
        "structured_job": structured_job
    }

# Node 4: Compare Resume to JD
def compare_resume_to_jd(state: ResumeMatchState):
    compare_prompt = f"""
    Compare this structured resume to the job description and give a match score (0-100), 
    along with reasons for the score:

    Resume:\n{state['structured_resume']}\n
    Job:\n{state['structured_job']}
    """

    result = llm.invoke([HumanMessage(content=compare_prompt)]).content

    # Extract score and feedback
    import re
    score_match = re.search(r"\b(\d{1,3})\b", result)
    score = int(score_match.group(1)) if score_match else 50

    return {
        "match_score": score,
        "match_feedback": result
    }

# Node 5a: Generate Cover Letter
def generate_cover_letter(state: ResumeMatchState):
    prompt = f"""
    Based on the resume and job description below, write a tailored cover letter:

    Resume:\n{state['structured_resume']}\n
    Job:\n{state['structured_job']}
    """
    letter = llm.invoke([HumanMessage(content=prompt)]).content
    return {"cover_letter": letter}

# Node 5b: Suggest Improvements
def suggest_resume_improvements(state: ResumeMatchState):
    prompt = f"""
    The resume scored {state['match_score']} when matched to this job. Suggest specific improvements 
    to increase the score:

    Resume:\n{state['structured_resume']}\n
    Job:\n{state['structured_job']}
    """
    suggestions = llm.invoke([HumanMessage(content=prompt)]).content
    return {"suggestions": suggestions}

# Router to decide which path to follow based on score
def score_router(state: ResumeMatchState):
    if state["match_score"] >= 70:
        return "generate_cover_letter"
    else:
        return "suggest_improvements"

# ----------- Build the Graph -----------
builder = StateGraph(ResumeMatchState)

builder.add_node("load_resume", load_resume)
builder.add_node("load_job_description", load_job_description)
builder.add_node("extract_info", extract_info)
builder.add_node("compare", compare_resume_to_jd)
builder.add_node("generate_cover_letter", generate_cover_letter)
builder.add_node("suggest_improvements", suggest_resume_improvements)

builder.set_entry_point("load_resume")
builder.add_edge("load_resume", "load_job_description")
builder.add_edge("load_job_description", "extract_info")
builder.add_edge("extract_info", "compare")
builder.add_conditional_edges("compare", score_router, {
    "generate_cover_letter": "generate_cover_letter",
    "suggest_improvements": "suggest_improvements"
})
builder.add_edge("generate_cover_letter", END)
builder.add_edge("suggest_improvements", END)

app = builder.compile()


In [4]:
inputs = {
    "resume_text": """
Umesh Gopal
📱 +91 88 90970695 | 📧 umesh.gopal@zetymail.in
🔗 linkedin.com/in/umesh.gopal
📍 Ludhiana, Punjab, India

🔹 Professional Summary
Experienced Software Developer with 6+ years of expertise in building and maintaining scalable web applications. Proven track record in Angular, TypeScript, Java, and cloud-native development using Azure and Docker. Strong background in full-stack solutions, DevOps practices, and team leadership. Passionate about user-centric design, high-performance systems, and agile collaboration. Adept at mentoring junior developers and driving continuous improvement in cloud-based environments.

🔹 Key Skills
Frontend: Angular (v2–v16), TypeScript, Angular Material, HTML5, CSS3, RxJS
Backend: Java, Python, Node.js, RESTful APIs, SQL
Cloud & DevOps: Azure (App Services, Functions, Storage), Docker, GitLab CI/CD, Git, Azure DevOps
Practices: Agile/Scrum, CI/CD, SDLC, Code Reviews, Technical Documentation, UX Writing
Soft Skills: Mentoring, Problem-solving, Stakeholder Communication, Team Collaboration

🔹 Professional Experience
Senior Frontend Developer
Naposoft – Ludhiana, Punjab
📆 Jan 2022 – Present

Led front-end architecture and development for cloud-hosted applications using Angular 14+, Azure, and Docker.

Mentored 4 junior developers, improving code quality and delivery timelines by 30%.

Collaborated with cross-functional teams to implement scalable UI solutions and optimize performance.

Implemented CI/CD pipelines with GitLab for automated testing, linting, and deployments.

Worked closely with internal customers to gather feedback, iterate on UI designs, and drive user satisfaction.

Key Contributions:
✅ Migrated legacy AngularJS apps to Angular 16 with full testing coverage.
✅ Spearheaded accessibility improvements that increased usability scores by 25%.

Full-Stack Developer
Naposoft – Ludhiana, Punjab
📆 Feb 2019 – Dec 2021

Developed and maintained RESTful APIs in Java and integrated with Angular frontends.

Wrote automation scripts and unit tests to improve deployment and release cycles.

Participated in cloud deployment workflows using Docker and Azure App Services.

Regularly collaborated with QA and design teams to ship high-quality features.

Key Contributions:
✅ Reduced UI-related bugs by 40% through improved component modularization and testing.
✅ Automated analytics reporting for product usage, aiding data-driven decisions.

🔹 Projects (Highlighted)
Cloud Task Manager App

End-to-end web application built with Angular 16, integrated with Azure Functions and SQL backend.

Dockerized and deployed using GitLab CI/CD and Azure App Services.

UX Review Blog & Community

Authored UX case studies on 25+ apps, driving 5,000+ views and community discussion.

Published UI/UX redesign concepts, contributing to open-source design feedback.

🔹 Education
Bachelor of Science in Computer Engineering
Guru Nanak Dev Polytechnic College, Ludhiana
📆 Sep 2015 – Apr 2018 | 🎓 CGPA: 88%

🔹 Certifications
Oracle Certified Foundations Associate – Java, Oracle, Oct 2021

Frontend Development with Angular, Coursera, 2024

Microsoft Certified: Azure Developer Associate, Microsoft, 2024

🔹 Languages
English – Advanced | Hindi – Native | Punjabi – Native

🔹 Additional Information
Interests: UX/UI design, app performance analysis, contributing to open-source
Currently Exploring: Azure Bicep, Kubernetes, Advanced RxJS patterns



""",
    "job_description": """

The Competence Center for Cloud Adoption and Custom Development specializes in operating, integrating, and continuously optimizing various corporate services using public cloud solutions. We emphasize cross-functional collaboration and agile principles to deliver comprehensive end-to-end services with a focus on customer satisfaction.

(Senior) Front/Full-stack Developer, your role, grounded in a solid software development background and a proactive approach, will be crucial in advancing our cloud-based solutions. You'll ensure these solutions meet the evolving needs of our internal customers and continually seek and implement optimizations in collaboration with the team and stakeholders

 

What You Will Do 
Lead and oversee the development, implementation, and maintenance of front-end applications, utilizing a modern technology stack including Angular, TypeScript, Docker, GitLab, and Azure Cloud.
Drive collaboration with team members and internal customers to design, adapt, and optimize custom-developed cloud solutions for diverse projects, including robust public cloud services (IaaS/PaaS).
Mentor junior developers in web development, continuously enhancing technical skills and knowledge with a focus on innovation and emerging technologies.
Utilize and mentor others in various technologies while emphasizing on customer satisfaction and high-quality delivery.
 
What You Bring
Expertise in Angular (minimum 2 years, proficient in the latest version), Angular Material, and TypeScript
Autonomous, quality-focused work attitude, capable of leading and mentoring with a strong emphasis on customer satisfaction and high-quality delivery
Excellent communication skills in English (both written and verbal); proficiency in German is an advantage.

"""
}

final_state = app.invoke(inputs)

print("✅ Match Score:", final_state.get("match_score"))
print("🧠 Feedback:\n", final_state.get("match_feedback"))

if final_state.get("cover_letter"):
    print("\n📄 Generated Cover Letter:\n", final_state["cover_letter"])
if final_state.get("suggestions"):
    print("\n✏️ Suggested Improvements:\n", final_state["suggestions"])


✅ Match Score: 95
🧠 Feedback:
 Match Score: 95

Reasoning:
The resume demonstrates a strong match with the job requirements. The candidate has extensive experience in frontend development using Angular, TypeScript, and Angular Material. They have also worked on backend development with Java and RESTful APIs, which aligns with the job's need for a modern technology stack.

The candidate's experience in leading and mentoring junior developers, as well as their background in Agile/Scrum and CI/CD practices, are valuable assets for the job responsibilities. Their proficiency in using Docker, GitLab, and Azure Cloud is another strong point in favor of the candidate.

The resume also highlights the candidate's soft skills, such as problem-solving and stakeholder communication, which are essential for collaborating with team members and internal customers. The candidate's interests in UX/UI design, app performance analysis, and open-source contributions show their dedication to continuous lea